Para construir un **canal endémico** metodológicamente correcto (utilizando el método clásico de los **cuartiles**, que es el estándar de oro en epidemiología adoptado por la OPS/OMS) necesitamos calcular los percentiles 25, 50 y 75 de los casos históricos acumulados por semana epidemiológica.

Posteriormente, utilizaremos los niveles del canal endémico para transformar el problema a una clasificación binaria. Definiremos que hay una **"Alerta epidemiológica" (Clase 1)** si los casos observados superan el Q3 (Percentil 75, Zona de Epidemia) y **"Normalidad/Seguridad" (Clase 0)** si están por debajo. A partir de allí, entrenamos la Regresión Logística usando las variables climáticas clave reducidas por Lasso.

Dado que indicas que la ubicación de estudio es **Caucasia - Antioquia**, este script asume que los datos pertenecen a dicho municipio, lee el dataset reducido de Lasso, genera el canal, entrena el modelo logístico para predecir la probabilidad de alarma y guarda la gráfica requerida.

Aquí tienes el script `.py` completo:

```python


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import warnings

# Silenciar advertencias
warnings.filterwarnings("ignore")

# 1. Configuración de rutas e infraestructura de directorios
ruta_dataset_reducido = r"C:\Users\marco\Documentos\investigacion\rolling_forecast\2_datos\2_procesados\dataset_reducido_lasso.xlsx"
directorio_resultados = r"C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados"
os.makedirs(directorio_resultados, exist_ok=True)

if not os.path.exists(ruta_dataset_reducido):
    raise FileNotFoundError(f"No se encontró el dataset reducido de Lasso en: {ruta_dataset_reducido}")

# 2. Cargar datos reducidos
df = pd.read_excel(ruta_dataset_reducido)
df['fecha'] = pd.to_datetime(df['fecha'])

# =============================================================================
# 3. CONSTRUCCIÓN DEL CANAL ENDÉMICO (Método de Cuartiles 2022-2025)
# =============================================================================
print("Calculando el Canal Endémico para Caucasia (Periodo Histórico 2022-2025)...")

# Filtrar el periodo de referencia solicitado para construir el canal
df_historico = df[(df['año'] >= 2022) & (df['año'] <= 2025)]

# Agrupar por semana epidemiológica para obtener los percentiles epidemiológicos
canal_mediana = df_historico.groupby('semana_epi')['casos_dengue'].quantile(0.50).rename('Q2_Mediana')
canal_q1 = df_historico.groupby('semana_epi')['casos_dengue'].quantile(0.25).rename('Q1_Exito')
canal_q3 = df_historico.groupby('semana_epi')['casos_dengue'].quantile(0.75).rename('Q3_Alerta')

# Consolidar el dataframe maestro del canal endémico (Semanas 1 a 53)
df_canal = pd.concat([canal_q1, canal_mediana, canal_q3], axis=1).reset_index()

# =============================================================================
# 4. PREPARACIÓN DE DATOS PARA LA REGRESIÓN LOGÍSTICA
# =============================================================================
print("Modelando variables exógenas y vector objetivo...")

# Unir los umbrales del canal endémico al dataset general mediante la semana epidemiológica
df_modelo = pd.merge(df, df_canal, on='semana_epi', how='left')

# Definir la variable objetivo binaria: 1 si está en Zona de Alerta/Epidemia (Supera Q3), 0 en caso contrario
df_modelo['alerta_epidemia'] = (df_modelo['casos_dengue'] > df_modelo['Q3_Alerta']).astype(int)

# Aislar las variables predictoras (todas las columnas reducidas por Lasso)
columnas_excluidas = ['fecha', 'año', 'semana_epi', 'casos_dengue', 'Q1_Exito', 'Q2_Mediana', 'Q3_Alerta', 'alerta_epidemia']
columnas_predictores = [col for col in df_modelo.columns if col not in columnas_excluidas]

X = df_modelo[columnas_predictores]
y = df_modelo['alerta_epidemia']

# Separar entrenamiento (pasado) y testeo (lo que va de 2026)
X_train = X[df_modelo['año'] < 2026]
y_train = y[df_modelo['año'] < 2026]

X_test = X[df_modelo['año'] == 2026]
y_test = y[df_modelo['año'] == 2026]

# Validar si el set de testeo cuenta con datos suficientes para evaluar
if len(X_test) == 0:
    print("[Aviso] No hay registros para el año 2026. Usando un split aleatorio del 80/20 para resguardar la ejecución.")
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Estandarizar los predictores climáticos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =============================================================================
# 5. AJUSTE Y EVALUACIÓN DE LA REGRESIÓN LOGÍSTICA
# =============================================================================
print("Entrenando clasificador probabilístico de Regresión Logística...")
modelo_log = LogisticRegression(random_state=42, class_weight='balanced')
modelo_log.fit(X_train_scaled, y_train)

# Calcular probabilidades de ocurrencia de brote/alerta epidémica
probabilidades_brote = modelo_log.predict_proba(X_test_scaled)[:, 1]
predicciones_binarias = modelo_log.predict(X_test_scaled)

print("\n=== RENDIMIENTO DE CLASIFICACIÓN (ZONA DE EPIDEMIA) ===")
print(classification_report(y_test, predicciones_binarias))
try:
    print(f"Área Bajo la Curva ROC (AUC): {roc_auc_score(y_test, probabilidades_brote):.4f}")
except ValueError:
    pass

# =============================================================================
# 6. GENERACIÓN DEL GRÁFICO DEL CANAL ENDÉMICO (2022-2025)
# =============================================================================
print("\nGenerando gráfica del canal endémico...")
plt.figure(figsize=(12, 6))

# Dibujar las bandas epidemiológicas tradicionales
plt.fill_between(df_canal['semana_epi'], 0, df_canal['Q1_Exito'], color='#c2e699', alpha=0.6, label='Zona de Éxito (< Q1)')
plt.fill_between(df_canal['semana_epi'], df_canal['Q1_Exito'], df_canal['Q2_Mediana'], color='#ffffcc', alpha=0.6, label='Zona de Seguridad (Q1 - Q2)')
plt.fill_between(df_canal['semana_epi'], df_canal['Q2_Mediana'], df_canal['Q3_Alerta'], color='#fecc5c', alpha=0.6, label='Zona de Alerta (Q2 - Q3)')
plt.fill_between(df_canal['semana_epi'], df_canal['Q3_Alerta'], df_canal['Q3_Alerta'].max() * 1.5, color='#f03b20', alpha=0.3, label='Zona de Epidemia (> Q3)')

# Graficar las líneas de demarcación
plt.plot(df_canal['semana_epi'], df_canal['Q1_Exito'], color='#78c679', linestyle=':', linewidth=1.5)
plt.plot(df_canal['semana_epi'], df_canal['Q2_Mediana'], color='#fe9929', linestyle='-', linewidth=2)
plt.plot(df_canal['semana_epi'], df_canal['Q3_Alerta'], color='#e31a1c', linestyle='--', linewidth=1.5)

# Títulos y formato del eje institucional
plt.title('Canal Endémico de Dengue en Caucasia - Antioquia\nPeriodo de Referencia Histórica (2022-2025)', fontsize=13, fontweight='bold')
plt.xlabel('Semana Epidemiológica', fontsize=11)
plt.ylabel('Casos de Dengue', fontsize=11)
plt.xlim(1, df_canal['semana_epi'].max())
plt.grid(axis='both', linestyle=':', alpha=0.5)

# Posicionar la leyenda fuera o dentro de forma limpia
plt.legend(loc='upper right', frameon=True, facecolor='white', shadow=True)

# Guardar la gráfica en la ruta de resultados especificada
ruta_grafico = os.path.join(directorio_resultados, "canal_endemico_caucasia.png")
plt.tight_layout()
plt.savefig(ruta_grafico, dpi=300)
plt.close()

print(f"[OK] Gráfico del canal endémico guardado en: {ruta_grafico}")

# =============================================================================
# 7. EXPORTACIÓN DE PROBABILIDADES DE 2026
# =============================================================================
df_salida_2026 = df_modelo[df_modelo['año'] == 2026].copy().reset_index(drop=True)
if len(df_salida_2026) > 0:
    df_salida_2026['Probabilidad_Alerta_Epidemia'] = probabilidades_brote
    df_salida_2026['Prediccion_Brote_Binaria'] = predicciones_binarias
    
    ruta_excel_salida = os.path.join(directorio_resultados, "probabilidad_alerta_2026.xlsx")
    df_salida_2026[['fecha', 'año', 'semana_epi', 'casos_dengue', 'Q3_Alerta', 'Probabilidad_Alerta_Epidemia', 'Prediccion_Brote_Binaria']].to_excel(ruta_excel_salida, index=False)
    print(f"[OK] Predicciones probabilísticas de alerta para 2026 guardadas en: {ruta_excel_salida}")


Calculando el Canal Endémico para Caucasia (Periodo Histórico 2022-2025)...
Modelando variables exógenas y vector objetivo...
Entrenando clasificador probabilístico de Regresión Logística...

=== RENDIMIENTO DE CLASIFICACIÓN (ZONA DE EPIDEMIA) ===
              precision    recall  f1-score   support

           0       0.83      0.62      0.71        16
           1       0.33      0.60      0.43         5

    accuracy                           0.62        21
   macro avg       0.58      0.61      0.57        21
weighted avg       0.71      0.62      0.65        21

Área Bajo la Curva ROC (AUC): 0.6750

Generando gráfica del canal endémico...
[OK] Gráfico del canal endémico guardado en: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\canal_endemico_caucasia.png
[OK] Predicciones probabilísticas de alerta para 2026 guardadas en: C:\Users\marco\Documentos\investigacion\rolling_forecast\3_resultados\probabilidad_alerta_2026.xlsx



```

### Explicación del Funcionamiento Técnico:

1. **Zonificación Epidemiológica:** El script calcula dinámicamente los límites del canal para Caucasia: **Éxito** ($<\text{P25}$), **Seguridad** ($\text{P25}-\text{P50}$), **Alerta** ($\text{P50}-\text{P75}$) y **Epidemia** ($>\text{P75}$).
2. **Uso Inteligente de las Dimensiones Lasso:** Mantiene intactas las variables climáticas filtradas por Lasso que el modelo ARIMAX usaba, pero ahora entrena una sigmoide logística $P(Y=1) = \frac{1}{1 + e^{-z}}$ para clasificar si los factores climáticos de las semanas previas van a empujar al municipio hacia una condición de alerta epidemiológica.
3. **Manejo del Desbalance:** Se añade el parámetro `class_weight='balanced'` en el clasificador lineal, debido a que las semanas de epidemia real suelen ser menos comunes que las semanas de normalidad, evitando que el modelo se sesgue hacia falsos negativos.